In [ ]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

# ----------------------------
# 1) Load prepared interactions
# ----------------------------
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"
PREP_ROOT = DATA_ROOT / "prepared"
FILE_PATH = PREP_ROOT / "interactions_prepared.csv"

df = pd.read_csv(FILE_PATH)

required_cols = {"user_id", "item_id", "event_weight"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Loaded file:", FILE_PATH)
print("Raw shape:", df.shape)

# ----------------------------
# 2) Aggregate duplicate interactions
# ----------------------------
df = (
    df.groupby(["user_id", "item_id"], as_index=False)["event_weight"]
      .sum()
)

user_counts = df.groupby("user_id")["item_id"].nunique()
eligible_users = user_counts[user_counts >= 2].index
df = df[df["user_id"].isin(eligible_users)].copy()

print("Filtered shape:", df.shape)
print("Unique users:", df["user_id"].nunique())
print("Unique items:", df["item_id"].nunique())

# ----------------------------
# 3) Leave-one-out split
# ----------------------------
df = df.sample(frac=1, random_state=42).copy()
test_idx = df.groupby("user_id").head(1).index

test_df = df.loc[test_idx].copy()
train_df = df.drop(test_idx).copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# ----------------------------
# 4) Build train matrix
# ----------------------------
user_ids = train_df["user_id"].unique()
item_ids = train_df["item_id"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_item = {i: it for it, i in item_to_idx.items()}

test_df = test_df[
    test_df["user_id"].isin(user_to_idx) & test_df["item_id"].isin(item_to_idx)
].copy()

rows = train_df["user_id"].map(user_to_idx)
cols = train_df["item_id"].map(item_to_idx)
vals = train_df["event_weight"].astype("float32")

user_item = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_to_idx), len(item_to_idx)),
    dtype=np.float32
)

print("User-item matrix shape:", user_item.shape)

# ----------------------------
# 5) Train SVD
# ----------------------------
min_dim = min(user_item.shape)
n_components = min(20, max(2, min_dim - 1))

svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(user_item).astype(np.float32)
item_factors = svd.components_.T.astype(np.float32)

print("Chosen n_components:", n_components)
print("Explained variance ratio sum:", round(svd.explained_variance_ratio_.sum(), 4))

# ----------------------------
# 6) Recommend without full score matrix
# ----------------------------
def recommend_svd(user_id, k=10):
    if user_id not in user_to_idx:
        return []

    uidx = user_to_idx[user_id]

    # Score only this user against all items
    scores = np.dot(user_factors[uidx], item_factors.T)

    seen_items = train_df.loc[train_df["user_id"] == user_id, "item_id"].tolist()
    seen_indices = [item_to_idx[item] for item in seen_items if item in item_to_idx]
    scores[seen_indices] = -np.inf

    top_idx = np.argsort(scores)[::-1][:k]
    return [idx_to_item[i] for i in top_idx if np.isfinite(scores[i])]

sample_user = train_df["user_id"].iloc[0]
print("Sample user:", sample_user)
print("Top-5 recommendations:", recommend_svd(sample_user, k=5))

# ----------------------------
# 7) Ranking metrics
# ----------------------------
def evaluate_ranking_metrics(test_df, recommend_fn, k=10):
    user_truth = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    precisions, recalls, ndcgs = [], [], []

    for user_id, true_items in user_truth.items():
        recs = recommend_fn(user_id, k=k)
        if not recs:
            continue

        hits = [1 if item in true_items else 0 for item in recs]
        hit_count = sum(hits)

        precision = hit_count / k
        recall = hit_count / len(true_items)

        dcg = sum(hit / np.log2(i + 2) for i, hit in enumerate(hits))
        ideal_hits = [1] * min(len(true_items), k)
        idcg = sum(hit / np.log2(i + 2) for i, hit in enumerate(ideal_hits))
        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        f"Precision@{k}": np.mean(precisions) if precisions else 0.0,
        f"Recall@{k}": np.mean(recalls) if recalls else 0.0,
        f"NDCG@{k}": np.mean(ndcgs) if ndcgs else 0.0,
        "Evaluated Users": len(precisions)
    }

metrics_5 = evaluate_ranking_metrics(test_df, recommend_svd, k=5)
metrics_10 = evaluate_ranking_metrics(test_df, recommend_svd, k=10)

print("\nSVD Metrics@5")
for metric, value in metrics_5.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")

print("\nSVD Metrics@10")
for metric, value in metrics_10.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")


Loaded file: C:\Users\barath\recomart-pipeline\data\prepared\interactions_prepared.csv
Raw shape: (2755641, 10)
Filtered shape: (1025679, 3)
Unique users: 288080
Unique items: 156487
Train shape: (737599, 3)
Test shape: (288080, 3)
User-item matrix shape: (288080, 132535)
Chosen n_components: 20
Explained variance ratio sum: 0.1471
Sample user: 684514
Top-5 recommendations: [np.int64(48030), np.int64(5675), np.int64(4001), np.int64(149382), np.int64(464731)]
